# Week 31 Graded Mini Project

##  Overview:

```text
In this project, you will perform sentiment analysis on a dataset of Twitter posts using Recurrent
Neural Networks (RNN). You will preprocess the data, conduct exploratory data analysis (EDA), build
an RNN model for sentiment classification, and evaluate the performance of your model. The goal is
to classify the sentiment of tweets as positive, negative, or neutral based on the text data.
```

## Dataset Overview:
```text
It contains tweet data, including the tweet text, sentiment labels (positive, negative, or neutral), and
other metadata (e.g., tweet ID, user information, and date of the tweet).
```

#### Part 1: Data Processing

##### 1. Load the Dataset:
- Load the CSV file into an appropriate data structure (e.g., DataFrame).

In [ ]:


# ================================
# CORE LIBRARIES
# ================================

import re

import pandas as pd
from IPython.display import display

# ================================
# LOAD DATASET
# ================================
# - Read CSV file
# - Assign column names (dataset has no headers)
# - Inspect initial rows to verify structure

twitter_data = pd.read_csv('data/twitter_training.csv')

display(twitter_data.head(5))


>On Observing the dataset, given data set does not contain headers, hence we will have to create a headers for the same


For example:

```python
columns = ['id', 'entity', 'sentiment', 'text']

twitter_data = pd.read_csv(
    "data/twitter_training.csv",
    names=columns
)

```

In [ ]:
columns = ['id', 'entity', 'sentiment', 'text']

twitter_data.columns = columns
display(twitter_data.head(5))

##### 2. Data Cleaning:
- Check for and handle missing values in the dataset.
- Remove duplicates if any exist.
- Perform text cleaning on tweet text (e.g., remove URLs, mentions, hashtags, special characters).
- Tokenize the text and convert words to lowercase.
- Remove stop words and apply stemming or lemmatisation.

In [ ]:
# ================================
# HANDLE MISSING VALUES
# ================================
# - Drop rows where sentiment is missing (mandatory)
# - Handle missing text:
#     → If very small % → drop
#     → Else → fill with placeholder
# - Ensure clean dataset before NLP processing

## Keep only the required columns
twitter_data = twitter_data[['sentiment', 'text']]

## Check for missing values
print(f"Before handling missing values: {twitter_data.shape}")
print(twitter_data.isnull().sum())

## Drop rows where sentiment is missing (mandatory)
twitter_data = twitter_data.dropna(subset=['sentiment'])

## For text
## If missing < 5 then drop
## Else fill
missing_text_pct = twitter_data['text'].isnull().mean() * 100

if missing_text_pct < 5:
    twitter_data = twitter_data.dropna(subset=['text'])
else:
    twitter_data['text'] = twitter_data['text'].fillna('Unknown')

print(f"After handling missing values: {twitter_data.shape}")
print(twitter_data.isnull().sum())

##### Missing Value Handling

###### Before Handling Missing Values
- Dataset Shape: **(74,681, 2)**

| Column     | Missing Values |
|------------|---------------|
| sentiment  | 0             |
| text       | 686           |

---

###### Action Taken
- Removed rows where **text is missing**, since text is the primary feature for sentiment analysis.
- Missing percentage in `text` column ≈ **0.9%**, which is negligible.

---

###### After Handling Missing Values
- Dataset Shape: **(73,995, 2)**

| Column     | Missing Values |
|------------|---------------|
| sentiment  | 0             |
| text       | 0             |

---

###### Justification
- Rows with missing text provide **no useful information** for NLP models.
- Dropping less than 1% of data ensures:
  - Better data quality
  - No significant loss of information
  - Improved model performance

---

In [ ]:
# ================================
# NLP PREPROCESSING PIPELINE
# ================================
# - Regex cleaning (URLs, mentions)
# - Emoji normalization (convert to sentiment tokens)
# - Tokenization using spaCy
# - Lemmatization + stopword removal
# - Preserve emoji sentiment signals
import spacy
import emoji

## Load spaCy model
nlp = spacy.load('en_core_web_sm')

## Precompile regex patters
URL_PATTERN = re.compile(r'http\S+')
MENTION_PATTERN = re.compile(r'@\w+|#\w+')
SPECIAL_PATTERN = re.compile(r'[^a-zA-Z\s]')

# ================================
# EMOJI NORMALIZATION
# ================================
# Convert ALL emojis into semantic categories

def replace_emojis(text):
    result = []

    for char in text:
        if char in emoji.EMOJI_DATA:
            name = emoji.demojize(char)

            if any(w in name for w in ["smile", "love", "heart", "joy", "laugh"]):
                result.append(" EMO_POS ")
            elif any(w in name for w in ["angry", "sad", "cry", "rage", "fear"]):
                result.append(" EMO_NEG ")
            else:
                result.append(" EMO_NEU ")
        else:
            result.append(char)

    return "".join(result)

def basic_clean(text):
    """
    Performs initial cleaning:
    - Removes URLs, mentions, hashtags
    - Converts emojis to meaningful words (important for sentiment)
    - Removes unwanted characters
    - Converts text to lowercase
    """
    text = URL_PATTERN.sub('', text)
    text = MENTION_PATTERN.sub('', text)

    # Convert emojis to text
    # Example: "I love this 😍" → "I love this smiling_face_with_heart_eyes"
    # Convert emojis ONLY if present and avoid long token noise
    # APPLY EMOJI NORMALIZATION
    text = replace_emojis(text)

    # Remove unwanted characters but keep emoji text
    text = SPECIAL_PATTERN.sub(' ', text)

    return text.lower()


def spacy_clean_pipeline(texts):
    """
    Advanced NLP pipeline:
    - Tokenization
    - Lemmatization
    - Stopword removal
    - Keeps emoji sentiment tokens
    """

    cleaned = []

    for doc in nlp.pipe(texts, batch_size=500):
        tokens = []

        for token in doc:

            # Keep words + emoji tokens
            if (token.is_alpha or token.text.lower() in {"emo_pos", "emo_neg", "emo_neu"}):

                if token.is_stop:
                    continue

                if len(token.text) > 2 or token.text.lower().startswith("emo"):

                    if token.text.lower().startswith("emo"):
                        tokens.append(token.text.lower())
                    else:
                        tokens.append(token.lemma_)


        cleaned.append(" ".join(tokens))

    # return AFTER processing all docs
    return cleaned

In [ ]:
# =====================================
# DATA CLEANING PIPELINE
# =====================================
# Step 1: Normalize sentiment labels (merge Irrelevant → Neutral)
# Step 2: Basic text cleaning (URLs, mentions, emojis)
# Step 3: Remove duplicate tweets
# Step 4: Apply NLP pipeline (tokenization + lemmatization)

# Step 1: Normalize sentiment labels (merge Irrelevant → Neutral)
twitter_data['sentiment'] = twitter_data['sentiment'].replace({
    'Irrelevant': 'Neutral'
})

## Step 2: Basic cleaning - Perform text cleaning on tweet text (e.g., remove URLs, mentions, hashtags, special characters).
twitter_data['text'] = twitter_data['text'].map(basic_clean)

## Step 3 Remove duplicates
print("Duplicates:", twitter_data.duplicated(subset=['text']).sum())
twitter_data = twitter_data.drop_duplicates(subset=['text']).reset_index(drop=True)

## Step 4: Tokenize the text and convert words to lowercase, and Remove stop words and apply stemming or lemmatization.
twitter_data['clean_text'] = spacy_clean_pipeline(twitter_data['text'].tolist())


In [ ]:
## Shape of the dataset after preprocessing
print(twitter_data.shape)

##### Part 2: Exploratory Data Analysis (EDA)
1. Basic Statistics:
    - Summarise the dataset (mean, median, mode, etc.).
    - Explore the distribution of tweet sentiments (e.g., how many positive, negative, and
neutral tweets are there?).
2. Visualisations:
    - Create visualisations to showcase:
        - The distribution of sentiments.
        - The frequency of top words in positive, negative, and neutral sentiments.
        - Word clouds for positive and negative tweets.
        - The relationship between tweet length and sentiment.

In [ ]:
# ================================
# BASIC DATA ANALYSIS
# ================================
# - Dataset size
# - Tweet length statistics
# - Sentiment distribution
# - Understand data balance and structure

print(f'Dataset shape {twitter_data.shape}')
twitter_data.head(100000)

# Summary statistics for tweet length
twitter_data['length'] = twitter_data['clean_text'].apply(lambda x: len(x.split()))

# Length Distribution
print("\n Length Distribution")
print(f'Mean {twitter_data.length.mean()}')
print(f'Median {twitter_data.length.median()}')
print(f'Mode {twitter_data.length.mode()[0]}')

# Sentiment Distribution
# Count of each sentiment
sentiment_counts = twitter_data.sentiment.value_counts()

# Percentage distribution
sentiment_percent = (sentiment_counts / len(twitter_data)) * 100

print("\nSentiment Percentage:")
print(sentiment_percent.round(2))

In [ ]:
# ================================
# VISUALIZATION
# ================================
# - Sentiment distribution plot
# - Top words per sentiment
# - Word clouds
# - Tweet length vs sentiment
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='sentiment', data=twitter_data)
plt.title('Sentiment Distribution')
plt.show()

In [ ]:
# =========================================================================
# The frequency of top words in positive, negative, and neutral sentiments.
# =========================================================================

from collections import Counter

for sentiment in twitter_data.sentiment.unique():
    text = " ".join(twitter_data[twitter_data.sentiment == sentiment].clean_text)
    words = text.split()

    common_words = Counter(words).most_common(10)
    words, counts = zip(*common_words)

    color_map = {
    'Positive': 'lightgreen',
    'Negative': 'red',
    'Neutral': 'blue'
}

    plt.bar(words, counts, color=color_map.get(sentiment, 'purple'))
    plt.xticks(rotation=45)
    plt.title(f'Top words {sentiment}')
    plt.show()

In [ ]:
# =============================================
# Word clouds for positive and negative tweets.
# =============================================
from wordcloud import WordCloud
positive_feedback = " ".join(twitter_data[twitter_data.sentiment == 'Positive'].clean_text)
pwc = WordCloud(width=800, height=400).generate(positive_feedback)

plt.imshow(pwc)
plt.axis('off')
plt.show()

negative_feedback = " ".join(twitter_data[twitter_data.sentiment == 'Negative'].clean_text)
nwc = WordCloud(width=800, height=400).generate(negative_feedback)

plt.imshow(nwc)
plt.axis('off')
plt.show()

In [ ]:
# ====================================================
# The relationship between tweet length and sentiment.
# ====================================================

sns.boxplot(x='sentiment', y='length', data=twitter_data)
plt.title('Tweet Length vs Sentiment')
plt.show()

3. Insights

In [ ]:
# ================================
# AUTO CALCULATIONS FOR REPORT
# ================================

# Dataset size
total_records = twitter_data.shape[0]

# Sentiment distribution
sentiment_counts = twitter_data['sentiment'].value_counts()

dominant_class = sentiment_counts.idxmax()
least_class = sentiment_counts.idxmin()

# Check imbalance
imbalance_ratio = sentiment_counts.max() / sentiment_counts.min()

if imbalance_ratio < 1.5:
    balance_type = "balanced"
elif imbalance_ratio < 3:
    balance_type = "slightly imbalanced"
else:
    balance_type = "highly imbalanced"

# Length stats
mean_len = round(twitter_data['length'].mean(), 2)
median_len = int(twitter_data['length'].median())

# Top words (overall)
from collections import Counter
all_words = " ".join(twitter_data['clean_text']).split()
top_words = [w for w, _ in Counter(all_words).most_common(5)]

# Top words per sentiment
sentiment_words = {}

for sentiment in twitter_data['sentiment'].unique():
    text = " ".join(twitter_data[twitter_data['sentiment'] == sentiment]['clean_text'])
    words = text.split()
    sentiment_words[sentiment] = [w for w, _ in Counter(words).most_common(3)]

In [ ]:
# ================================
# FINAL REPORT TEXT
# ================================

report = f"""
The exploratory data analysis provided several key insights into the dataset:

1. The dataset consists of {total_records} records and exhibits a {balance_type} distribution of sentiments,
   with '{dominant_class}' being the most frequent and '{least_class}' being the least frequent.

2. Tweets are generally short in length, with an average of {mean_len} words and a median of {median_len} words,
   which aligns with the nature of Twitter data.

3. Frequent word analysis shows commonly occurring terms such as {', '.join(top_words)}, indicating dominant themes in the dataset.

4. Distinct word patterns were observed across sentiment categories:
"""

# Add sentiment-specific words
for sentiment, words in sentiment_words.items():
    report += f"\n   - {sentiment} tweets commonly include words like {', '.join(words)}."

report += """

5. Word cloud visualizations further reinforce these patterns by highlighting sentiment-specific vocabulary.

6. The relationship between tweet length and sentiment suggests that different sentiments may exhibit variations
   in expression length, although most tweets remain relatively short.

Overall, the dataset shows clear linguistic patterns across sentiments, making it suitable for training
a deep learning model for sentiment classification.
"""

print(report)

##### Part 3: Building the RNN Model
1. Model Architecture:
    - Build an RNN model using LSTM (Long Short-Term Memory) or GRU (Gated Recurrent Units) for sentiment classification.
    - Use an embedding layer to represent the text data.
2. Model Implementation:
    - Split the dataset into training and testing sets.
    - Train the RNN model using the training set and evaluate using the test set.
    - Implement dropout and batch normalisation (if necessary) to improve model performance.
3. Evaluation:
    - Evaluate the performance of your RNN model using metrics such as accuracy, precision, recall, and F1-score.
    - Plot learning curves to monitor training progress and avoid overfitting.
    - Perform hyperparameter tuning (e.g., number of layers, hidden units, learning rate).
4. Model Improvement:
    - Implement techniques such as grid search, cross-validation, or transfer learning to improve model performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, GRU, Bidirectional,
    Dense, Dropout, Layer
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam

import tensorflow as tf
import numpy as np
import random

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [ ]:
# ================================
# TOKENIZATION + SEQUENCE PREPARATION
# ================================
# - Build vocabulary using tokenizer
# - Remove rare words (freq < threshold)
# - Convert text → sequences
# - Pad sequences to fixed length
# - Prepare input for model

## Initial Tokenizer
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(twitter_data['clean_text'])

# Word frequencies
word_counts = tokenizer.word_counts

# Remove rare words (frequency < 3)
threshold = 3
filtered_words = [w for w, c in word_counts.items() if c >= threshold]


# Dynamic vocabulary size
num_words = len(filtered_words) + 1  # +1 for OOV
print(f'Number of words: {num_words}')

# Reinitialize tokenizer
tokenizer = Tokenizer(oov_token='<OOV>', num_words=num_words)
tokenizer.fit_on_texts(twitter_data['clean_text'])

sequences = tokenizer.texts_to_sequences(twitter_data['clean_text'])

# Use 95th percentile (best practice)
max_len = int(twitter_data['length'].quantile(0.95))
print(f'Max length: {max_len}')

# Padding sequences
X = pad_sequences(sequences, maxlen=max_len, padding='post')

In [ ]:
# ===============
# # Encoding sentiment labels into numeric form
# ===============

le = LabelEncoder()
y = le.fit_transform(twitter_data['sentiment'])

In [ ]:
# ================================
# # Removing duplicate sequences to prevent data leakage between train and test
# ================================

import numpy as np

# Remove duplicates ONLY based on X (ignore label)
X_unique, idx = np.unique(X, axis=0, return_index=True)

# Split back
y = y[idx]
X = X_unique

print("After removing sequence duplicates:", X.shape)

In [ ]:
assert len(X) == len(y), "Mismatch between X and y!"

In [ ]:
# ================================
# Proper Train / Validation / Test Split
# ================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

In [ ]:
# ================================
# Verifying no data leakage between train and test sets
# ================================
print("Train/Test overlap:",
      len(set(map(tuple, X_train)) & set(map(tuple, X_test))))

In [ ]:
# ===============================
# Simple Attention Layer
# ===============================

class Attention(Layer):
    """
    Attention Layer:
    Assigns importance weights to each word in the sequence
    Helps model focus on relevant words for sentiment prediction
    """
    def build(self, input_shape):
        self.W = self.add_weight(
            shape=(input_shape[-1], 1),
            initializer='random_normal',
        )

    def call(self, x):
        # Score each word
        scores = tf.matmul(x, self.W)

        # Normalize using softmax
        weights = tf.nn.softmax(scores, axis=1)

        # Apply attention
        weighted = x * weights

        # Reduce to single vector
        return tf.reduce_sum(weighted, axis=1)



In [ ]:
# Compute class weights based on training labels
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary format (required by Keras)
class_weights = dict(enumerate(class_weights))

print("Class Weights:", class_weights)

In [ ]:
# ================================
# MODEL ARCHITECTURE (GRU + ATTENTION)
# ================================
# - Embedding layer → word representation
# - Bidirectional GRU → context learning
# - Attention → focus on important words
# - Dropout → prevent overfitting
# - Dense → feature extraction
# - Output → 3-class classification
# ============================================================
# Builds model dynamically based on hyperparameters
# ============================================================

def build_model(gru_units, dropout_rate, learning_rate):

    input_layer = Input(shape=(max_len,))

    # Embedding layer
    # Converts words into dense vector representations
    embedding = Embedding(input_dim=num_words, output_dim=200)(input_layer)

    # Bidirectional GRU (return sequences for attention)
    # Captures context from both forward and backward directions
    # Dropout helps prevent overfitting during sequence learning
    gru = Bidirectional(
        GRU(
            gru_units,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    )(embedding)

    # Attention layer
    # Assigns importance weights to words and creates a context vector
    attention = Attention()(gru)

    # Dropout after Attention layer
    # Reduces over-reliance on specific words/features (sequence-level regularization)
    drop = Dropout(dropout_rate)(attention)

    # Dense layer
    # Learns higher-level abstract features from attention output
    dense = Dense(64, activation='relu')(drop)

    # Output layer
    # 3 neurons → 3 sentiment classes (Negative, Neutral, Positive)
    # Softmax converts outputs into probability distribution
    output = Dense(3, activation='softmax')(dense)

    # Build model
    model = Model(inputs=input_layer, outputs=output)

    # Model Compile
    # Adam optimizer with standard learning rate for stable and fast convergence
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D


def build_model_mha(gru_units, dropout_rate, learning_rate):

    input_layer = Input(shape=(max_len,))

    # Embedding
    embedding = Embedding(input_dim=num_words, output_dim=200)(input_layer)

    # Bi-GRU
    gru = Bidirectional(
        GRU(
            gru_units,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    )(embedding)

    # ============================
    # Multi-Head Attention
    # ============================
    mha = MultiHeadAttention(
        num_heads=4,      # try 4 or 8
        key_dim=64
    )(gru, gru)

    # Residual + Normalization (VERY IMPORTANT)
    layer_norm = LayerNormalization()(gru + mha)

    # Global pooling (reduce sequence → vector)
    global_pooling = GlobalAveragePooling1D()(layer_norm)

    # Dropout
    drop_out = Dropout(dropout_rate)(global_pooling)

    # Dense
    dense = Dense(64, activation='relu')(drop_out)

    # Output
    output = Dense(3, activation='softmax')(dense)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
# =====================================
# MODEL (BIDIRECTIONAL GRU + ATTENTION)
# =====================================
model = build_model(gru_units=64, dropout_rate=0.5, learning_rate=0.001)
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    patience=1,
    factor=0.5
)
# Save best model based on validation LOSS
checkpoint_loss = ModelCheckpoint(
    filepath='best_model_loss.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Save best model based on validation ACCURACY
checkpoint_acc = ModelCheckpoint(
    filepath='best_model_accuracy.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

In [ ]:
from typing import cast
class TrainingMonitor(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        train_acc = cast(float, logs.get("accuracy") or 0.0)
        val_acc = cast(float, logs.get("val_accuracy") or 0.0)
        train_loss = cast(float, logs.get("loss") or 0.0)
        val_loss = cast(float, logs.get("val_loss") or 0.0)

        print("\n🔍 Epoch Summary")
        print(f"Epoch {epoch+1}")

        ta = f"{float(train_acc):.4f}" if train_acc is not None else "NA"
        va = f"{float(val_acc):.4f}" if val_acc is not None else "NA"
        tl = f"{float(train_loss):.4f}" if train_loss is not None else "NA"
        vl = f"{float(val_loss):.4f}" if val_loss is not None else "NA"

        print(f"Train Acc: {ta} | Val Acc: {va}")
        print(f"Train Loss: {tl} | Val Loss: {vl}")

        # Overfitting detection
        if train_acc is not None and val_acc is not None:
            if float(train_acc) - float(val_acc) > 0.05:
                print("⚠️ Overfitting detected")

        # Underfitting detection
        if train_acc is not None:
            if float(train_acc) < 0.6:
                print("⚠️ Still learning (underfitting)")

        if val_loss is not None:
            print("✅ Training stable")

In [ ]:
class SmartEarlyStop(tf.keras.callbacks.Callback):
    def __init__(self, threshold=0.06, patience=2):
        super().__init__()
        self.threshold = threshold
        self.patience = patience
        self.counter = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        train_acc = float(logs.get("accuracy", 0.0))
        val_acc = float(logs.get("val_accuracy", 0.0))

        gap = train_acc - val_acc

        print(f"📊 Gap (Train-Val): {gap:.4f}")

        # Overfitting detected
        if gap > self.threshold:
            self.counter += 1
            print(f"⚠️ Overfitting detected ({self.counter}/{self.patience})")

            if self.counter >= self.patience:
                print("🛑 Stopping training early due to overfitting!")
                self.model.stop_training = True
        else:
            self.counter = 0  # reset if healthy

In [ ]:
# ================================
# MODEL TRAINING
# ================================
# - EarlyStopping → stop when no improvement
# - ReduceLROnPlateau → adjust learning rate
# - ModelCheckpoint → save best models
# - Class weights → handle imbalance
# - SmartEarlyStop → detect overfitting early

history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler, checkpoint_loss, checkpoint_acc, TrainingMonitor(), SmartEarlyStop(threshold=0.08)],
    class_weight=class_weights,
    verbose=1
)

In [ ]:
# ================================
# MODEL EVALUATION
# ================================
# - Generate predictions on test data
# - Compute classification metrics
# - Analyze precision, recall, F1-score

y_pred = model.predict(X_test, verbose=0)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes,
    target_names=le.classes_
))

In [ ]:
# ============================================================
# LEARNING CURVES
# ============================================================
# Plotting training and validation performance to:
# - Monitor model learning behavior
# - Detect overfitting or underfitting
# - Identify optimal stopping point
# ============================================================

# Accuracy Curve
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Loss Curve
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# ================================
# CONFUSION MATRIX - GRU + Attention
# ================================
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('Confusion Matrix - GRU + Attention')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ============================================================
# CUSTOM PREDICTION (WORD / SENTENCE INPUT)
# ============================================================
# Allows user to input words/sentences and get predicted sentiment
# Useful for demonstrating model behavior
# ============================================================
from tensorflow.keras.models import load_model
def predict_sentiment(text_list, model):
    """
    Predict sentiment for given list of words/sentences
    """
    # Step 1: Basic cleaning
    cleaned = [basic_clean(text) for text in text_list]

    # Step 2: spaCy processing
    cleaned = spacy_clean_pipeline(cleaned)

    # Step 3: Convert to sequences
    seq = tokenizer.texts_to_sequences(cleaned)

    # Step 4: Padding
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    # Step 5: Predict
    preds = model.predict(padded, verbose=0)
    pred_classes = preds.argmax(axis=1)

    # Step 6: Convert back to labels
    labels = le.inverse_transform(pred_classes)

    # Print results
    for i, (text, label) in enumerate(zip(text_list, labels)):
        confidence = max(preds[i])
        print(f"Text: {text}")
        print(f"Predicted Sentiment: {label} (Confidence: {confidence:.2f})")
        print("-" * 50)

In [ ]:
test_samples = [
    "I absolutely love this game",
    "This is the worst experience ever",
    "The game is just average but good one, though",
    "Company announced a new update today"
]

best_model_loss = load_model(
    'best_model_loss.keras',
    custom_objects={'Attention': Attention}
)

best_model_acc = load_model(
    'best_model_accuracy.keras',
    custom_objects={'Attention': Attention}
)
predict_sentiment(test_samples, best_model_loss)
print("=" * 50)
predict_sentiment(test_samples, best_model_acc)

In [ ]:
model_mha = build_model_mha(64, 0.5, 0.001)

history_mha = model_mha.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler, TrainingMonitor(), SmartEarlyStop(threshold=0.08)],
    class_weight=class_weights
)

In [ ]:
y_pred_mha = model_mha.predict(X_test, verbose=0)
y_pred_classes_mha = y_pred_mha.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes_mha,
    target_names=le.classes_
))

In [ ]:
# ============================================================
# LEARNING CURVES
# ============================================================
# Plotting training and validation performance to:
# - Monitor model learning behavior
# - Detect overfitting or underfitting
# - Identify optimal stopping point
# ============================================================

# Accuracy Curve
plt.plot(history_mha.history['accuracy'], label='Train Accuracy')
plt.plot(history_mha.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Loss Curve
plt.plot(history_mha.history['loss'], label='Train Loss')
plt.plot(history_mha.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# ================================
# CONFUSION MATRIX - MHA
# ================================
cm_mha = confusion_matrix(y_test, y_pred_classes_mha)

plt.figure(figsize=(6,5))
sns.heatmap(cm_mha, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.title('Confusion Matrix - MultiHeadAttention')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ================================
# MODEL COMPARISON CHART
# ================================
from sklearn.metrics import accuracy_score

acc_gru = accuracy_score(y_test, y_pred_classes)
acc_mha = accuracy_score(y_test, y_pred_classes_mha)

model_names = ['GRU + Attention', 'MultiHeadAttention']
accuracies = [acc_gru, acc_mha]

plt.figure(figsize=(6,4))
plt.bar(model_names, accuracies)
plt.title('Model Comparison (Test Accuracy)')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.show()

The MultiHeadAttention model was implemented as an advanced architecture.
However, it resulted in lower performance compared to the GRU + Attention model.
This is likely due to the short length of tweets, where simpler sequential models
are more effective than complex attention mechanisms. Therefore, the GRU + Attention
model was selected as the final model.

In [ ]:
# ================================
# HYPERPARAMETER TUNING
# ================================
# - Try different combinations of:
#     → GRU units
#     → Dropout rate
#     → Learning rate
# - Use validation accuracy to select best config
# - Prevent overfitting using early stopping

from itertools import product

param_grid = {
    'gru_units': [64, 128],
    'dropout_rate': [0.3, 0.5],
    'learning_rate': [0.001, 0.0005],
}

results = []

for gru_units, dropout_rate, learning_rate in product(
    param_grid['gru_units'],
    param_grid['dropout_rate'],
    param_grid['learning_rate']
):
    print(f"Testing → GRU={gru_units}, Dropout={dropout_rate}, Learning Rate={learning_rate}")
    model_tune = build_model(gru_units, dropout_rate, learning_rate)



    early_stoping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
    )

    lr_schedulers = ReduceLROnPlateau(
        monitor='val_loss',
        patience=1,
        factor=0.5
    )

    history = model_tune.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stoping, lr_schedulers, SmartEarlyStop(threshold=0.08)],
        class_weight=class_weights,
        verbose=0
    )

    best_val_acc = max(history.history['val_accuracy'])

    results.append({
        "GRU": gru_units,
        "Dropout": dropout_rate,
        "LR": learning_rate,
        "Val Accuracy": best_val_acc
    })

# Convert to DataFrame
results_df = pd.DataFrame(results).sort_values(by="Val Accuracy", ascending=False)

print(results_df)

In [ ]:
best_config = results_df.iloc[0]
print("\nBest Configuration:")
print(best_config)

In [ ]:
best_model = build_model(
    int(best_config['GRU']),
    best_config['Dropout'],
    best_config['LR']
)

history_best = best_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler, checkpoint_loss, checkpoint_acc, TrainingMonitor(), SmartEarlyStop(threshold=0.08)],
    class_weight=class_weights,
    verbose=1
)

In [ ]:
y_pred_best = best_model.predict(X_test)
y_pred_classes_best = y_pred_best.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes_best,
    target_names=le.classes_
))

Hyperparameter tuning was conducted using different combinations of GRU units, dropout rates, and learning rates.
The best configuration achieved was GRU=64, dropout=0.3, and learning rate=0.001, which resulted in the highest validation accuracy.

Although configurations with GRU=128 showed comparable validation performance, they exhibited higher overfitting as indicated by a larger train-validation gap and did not improve test accuracy.

The selected configuration (GRU=64, dropout=0.3, learning rate=0.001) provided the best balance between performance and generalization, achieving improved test accuracy and stable training behavior. Therefore, this configuration was chosen as the final model.

In [ ]:
test_samples = [
    "I absolutely love this game",
    "This is the worst experience ever",
    "The game is just average but good one, though",
    "Company announced a new update today"
]


predict_sentiment(test_samples, best_model)

##### 4. Model Improvement:
- Implement techniques such as grid search, cross-validation, or transfer learning to
improve model performance.

In [ ]:
# ================================
# TRANSFER LEARNING (GLOVE)
# ================================
# - Load pretrained embeddings
# - Build embedding matrix
# - Use embeddings in model (non-trainable)
# - Evaluate performance vs baseline

embedding_index = {}
with open('glove/glove.twitter.27B.100d.txt', encoding='utf-8') as file:
    for line in file:
        parts = line.strip().split()

        # Skip if bad lines
        if len(parts) < 101:
            continue

        words = parts[0]
        vector = np.asarray(parts[1:], dtype='float32')
        embedding_index[words] = vector

print(f"Loaded {len(embedding_index)} word vectors from GloVe")

In [ ]:
# ============================================================
# CREATE EMBEDDING MATRIX
# ============================================================

embedding_dim = 100

embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < num_words:
        vector = embedding_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

In [ ]:
# ============================================================
# MODEL BUILDER FUNCTION - GLOVE
# ============================================================
# Builds model dynamically based on hyperparameters
# ============================================================

def build_model_with_glove(gru_units, dropout_rate, learning_rate):

    input_layer = Input(shape=(max_len,))

    # Embedding layer
    # USE PRETRAINED EMBEDDING
    embedding = Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True,
    )(input_layer)

    # Bidirectional GRU (return sequences for attention)
    # Captures context from both forward and backward directions
    # Dropout helps prevent overfitting during sequence learning
    gru = Bidirectional(
        GRU(
            gru_units,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    )(embedding)

    # Attention layer
    # Assigns importance weights to words and creates a context vector
    attention = Attention()(gru)

    # Dropout after Attention layer
    # Reduces over-reliance on specific words/features (sequence-level regularization)
    drop = Dropout(dropout_rate)(attention)

    # Dense layer
    # Learns higher-level abstract features from attention output
    dense = Dense(64, activation='relu')(drop)

    # Output layer
    # 3 neurons → 3 sentiment classes (Negative, Neutral, Positive)
    # Softmax converts outputs into probability distribution
    output = Dense(3, activation='softmax')(dense)

    # Build model
    model = Model(inputs=input_layer, outputs=output)

    # Model Compile
    # Adam optimizer with standard learning rate for stable and fast convergence
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model_glove = build_model_with_glove(64, 0.5, 0.001)

model_glove.summary()

In [ ]:
history_glove = model_glove.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler, checkpoint_loss, checkpoint_acc, TrainingMonitor(), SmartEarlyStop(threshold=0.08)],
    class_weight=class_weights,
    verbose=1
)

In [ ]:

y_pred_glove = model_glove.predict(X_test)
y_pred_classes_glove = y_pred_glove.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes_glove,
    target_names=le.classes_
))

Although Twitter-specific GloVe embeddings were used, the model performance remained significantly lower than the baseline model.

This can be attributed to the mismatch between pre-trained embeddings and the dataset-specific preprocessing pipeline, particularly the use of normalized emoji tokens (e.g., EMO_POS, EMO_NEG), which are not present in the GloVe vocabulary.

Additionally, the baseline model with trainable embeddings was better able to learn domain-specific patterns, including informal language and sentiment signals unique to the dataset.

Therefore, the tuned GRU + Attention model outperformed the GloVe-based model and was retained as the final model.

In [ ]:
# ================================
# CUSTOM EMBEDDINGS WORD2VEC
# ================================
# - Train embeddings on dataset
# - Build embedding matrix
# - Use embeddings in model
# - Compare with baseline and GloVe

from gensim.models import Word2Vec
import warnings
warnings.filterwarnings("ignore")

# Tokenize your cleaned text
sentences = [text.split() for text in twitter_data['clean_text']]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=100,   # embedding size
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

print("Word2Vec trained!")

In [ ]:
embedding_dim = 100

embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < num_words:
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]

In [ ]:
def build_model_with_w2v(gru_units, dropout_rate, learning_rate):

    input_layer = Input(shape=(max_len,))

    embedding = Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True
    )(input_layer)

    gru = Bidirectional(
        GRU(
            gru_units,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    )(embedding)

    attention = Attention()(gru)

    drop = Dropout(dropout_rate)(attention)
    dense = Dense(64, activation='relu')(drop)
    output = Dense(3, activation='softmax')(dense)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model_w2v = build_model_with_w2v(64, 0.5, 0.001)

history_w2v = model_w2v.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler, TrainingMonitor(), SmartEarlyStop(threshold=0.08)],
    class_weight=class_weights
)

In [ ]:
y_pred_w2v = model_w2v.predict(X_test)
y_pred_classes_w2v = y_pred_w2v.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes_w2v,
    target_names=le.classes_
))

Custom Word2Vec embeddings were trained on the dataset to capture domain-specific semantic relationships.

The Word2Vec-based model showed improved performance compared to GloVe, achieving better alignment with the dataset vocabulary. However, it still underperformed compared to the tuned GRU + Attention baseline model (accuracy ~0.86).

This can be attributed to the relatively smaller dataset size, which limits the effectiveness of Word2Vec in learning high-quality embeddings. Additionally, Word2Vec does not capture subword information, making it less effective for handling rare words and variations in informal Twitter text.

Therefore, while Word2Vec provided better domain adaptation than pre-trained embeddings, it was not sufficient to outperform the baseline model, which demonstrated superior generalization and accuracy.

In [ ]:
# ================================
# CUSTOM EMBEDDINGS FASTTEXT
# ================================
# - Train embeddings on dataset
# - Build embedding matrix
# - Use embeddings in model
# - Compare with baseline and GloVe

from gensim.models import FastText
import warnings
warnings.filterwarnings("ignore")

sentences = [text.split() for text in twitter_data['clean_text']]

ft_model = FastText(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

print("FastText trained!")

In [ ]:
embedding_dim = 100
embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < num_words:
        embedding_matrix[i] = ft_model.wv[word]  # 🔥 always works

In [ ]:
def build_model_with_fast_text(gru_units, dropout_rate, learning_rate):

    input_layer = Input(shape=(max_len,))

    embedding = Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True
    )(input_layer)

    gru = Bidirectional(
        GRU(
            gru_units,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    )(embedding)

    attention = Attention()(gru)

    drop = Dropout(dropout_rate)(attention)
    dense = Dense(64, activation='relu')(drop)
    output = Dense(3, activation='softmax')(dense)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model_fast_text = build_model_with_fast_text(64, 0.5, 0.001)

history_fast_text = model_fast_text.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, lr_scheduler, TrainingMonitor(), SmartEarlyStop(threshold=0.08)],
    class_weight=class_weights
)

In [ ]:

y_pred_fast_text = model_fast_text.predict(X_test)
y_pred_classes_fast_text = y_pred_fast_text.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_classes_fast_text,
    target_names=le.classes_
))

FastText embeddings were implemented to overcome the limitations of Word2Vec by incorporating subword information, allowing the model to better handle rare words and variations commonly found in informal Twitter text.

The FastText-based model demonstrated slight improvement over Word2Vec; however, it still did not surpass the performance of the tuned GRU + Attention baseline model (accuracy ~0.86).

Although FastText effectively captures subword-level patterns, the overall performance remained limited due to the dataset size and the already strong performance of the baseline model with trainable embeddings.

Therefore, the GRU + Attention model remained the best-performing approach, achieving higher accuracy and better generalization compared to both Word2Vec and FastText-based models.